# TIMidi — full fine-tune on GPU + submission

Fine-tunes the **given model** on the **full retain set** (no subsample) using the GPU,
then writes the 3 submission files and lets you download them.

**Before running:** `Runtime -> Change runtime type -> GPU (T4)`.

In [ ]:
# 1) code + data
!git clone -q https://github.com/julpfi/sapienza_tc_tim_machine_unlearning.git
%cd sapienza_tc_tim_machine_unlearning

In [ ]:
# 2) ---- CONFIG (tune these) ----
EPOCHS = 45
LR = 1e-3
OPT = "adam"          # "adam" or "sgd"
SCHED = "cosine"      # "cosine" or "none"
BATCH = 2048
SUBSAMPLE = 1.0       # 1.0 = full retain (GPU makes this cheap)
SUBMISSION_NAME = "TIMidi_gpu_v1"   # rename per attempt

In [ ]:
# 3) imports + GPU
import copy, time, glob
import numpy as np, pandas as pd, torch
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from pathlib import Path
from utils import functions as uf
from utils import unlearning as uu
from utils import eval as ue
from utils.model import DynamicMLP
from utils.submission import save_submission

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU!')
torch.manual_seed(42); np.random.seed(42)

In [ ]:
# 4) data split (same as main.py): forget / retain, then 85/15 train/val
df = pd.concat((pd.read_csv(f, sep=';') for f in glob.glob('./data/*c000.csv')), ignore_index=True)
fids = pd.read_csv('data/forget_data.csv')['user_id']
forget_df = df[df['user_id'].isin(fids)].reset_index(drop=True)
retain_df = df[~df['user_id'].isin(fids)].reset_index(drop=True)
train_df, val_df = train_test_split(retain_df, test_size=0.15, random_state=42)

Xtr, ytr, _, _ = uf.prepare_data(train_df, id_col='user_id', target_prefix='target__')
imp = SimpleImputer(strategy='median')
Xtr = imp.fit_transform(Xtr).astype(np.float32)
Xva, yva, _, _ = uf.prepare_data(val_df, id_col='user_id', target_prefix='target__')
Xva = imp.transform(Xva).astype(np.float32)
Xfo, yfo, _, _ = uf.prepare_data(forget_df, id_col='user_id', target_prefix='target__')
Xfo = imp.transform(Xfo).astype(np.float32)

pc = ytr.sum(0)
pw = torch.tensor((len(ytr) - pc) / (pc + 1e-6), device=device).clamp(0.1, 100.0).float()
print('train:', Xtr.shape)

In [ ]:
# 5) load the GIVEN model (start point) and optionally subsample the retain
payload = uf.load_pickle(Path('data') / 'model_artifact')
arch, best = payload['architecture'], payload['best_hyperparameters']
model = DynamicMLP(arch['input_dim'], arch['hidden_layers'], arch['num_outputs'])
model.load_state_dict(payload['state_dict'])

X_ft, y_ft = Xtr, ytr
if SUBSAMPLE < 1.0:
    keep = (ytr.sum(1) > 0) | (np.random.rand(len(ytr)) < SUBSAMPLE)
    X_ft, y_ft = Xtr[keep], ytr[keep]
print('fine-tune rows:', len(X_ft))

In [ ]:
# 6) fine-tune on GPU (TIMED = this is execution_time)
unlearned = copy.deepcopy(model)
start = time.perf_counter()
unlearned = uu.fine_tune(unlearned, X_ft, y_ft, pw, device,
                         epochs=EPOCHS, lr=LR, batch_size=BATCH, optimizer=OPT, sched=SCHED)
elapsed = time.perf_counter() - start
print(f'\nunlearning time: {elapsed:.1f}s')

In [ ]:
# 7) metrics on CPU (move model off GPU so eval + artifact are portable)
unlearned.to('cpu')
p10 = ue.precision_at_k(unlearned, Xva, yva)
auc = ue.mia_auc(unlearned, Xfo, yfo, Xva, yva)
print(f'local P@10 (blind proxy): {p10:.4f} | MIA {1-2*abs(auc-0.5):.4f} | time {elapsed:.1f}s')

In [ ]:
# 8) write submission + download
out = save_submission(unlearned, val_df, arch, best, elapsed, out_dir=SUBMISSION_NAME)
import shutil
from google.colab import files
shutil.make_archive(SUBMISSION_NAME, 'zip', SUBMISSION_NAME)
files.download(SUBMISSION_NAME + '.zip')
print('downloaded', SUBMISSION_NAME + '.zip  -> unzip, upload the FOLDER to Dropbox')

## Uso
1. Runtime GPU, poi *Esegui tutte*.
2. Scarica lo zip, **scompatta**, carica la **cartella** (coi 3 file) su Dropbox.
3. Per provare varianti: cambia `EPOCHS`/`SUBSAMPLE`/`LR` nella cella 2, cambia `SUBMISSION_NAME`, rilancia.

**Ricorda:** la P@10 locale e' un proxy cieco (~0.65). La precision vera la leggi solo sul leaderboard.